# 🧩 Mini-Lab: Local Backend

**Module: The Replaceable Model Interface** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Implement** a local AI backend using Ollama that conforms to the same `AIClient` protocol as the hosted backend
2. **Compare** hosted and local backends across five dimensions: cost, latency, privacy, setup, and RAM
3. **Explain** the tradeoffs between open (local) and closed (hosted) models for different use cases

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Open vs Closed Models | Open models are shared for you to run and modify; closed models are accessed via a provider with limited visibility or control. The choice affects cost, privacy, customization, and operational responsibility. |
| Interface Seam | A small typed contract between your app and an AI vendor. Your adapter returns your dataclass, never their SDK response object, so providers can be swapped without rewrites. |
| Baseline Comparison | Judge output against a simple reference baseline (a cheap, clear alternative), not against "looks good". If you cannot beat the baseline, you have not earned complexity. |


## Recap

In **mini-hosted-backend**, we built `HostedAIClient` — a backend that calls OpenAI's API over HTTP with a base URL, auth header, and timeout. It satisfies the `AIClient` protocol, just like our mock and echo backends.

Now we'll build **Backend 3: a local backend** that runs a model on your own machine via Ollama. Same protocol, same `AIRequest` in, same `AIResponse` out — but no API key, no network, and no per-token cost.

> **Prerequisite:** You need [Ollama](https://ollama.com) installed and a small model pulled (e.g., `ollama pull qwen2.5:0.5b`). If Ollama isn't available, the lab still works — we'll handle the missing-service case gracefully.

## Step 1 — Shared Types (Reused from Previous Labs)

We reuse the exact same `AIRequest`, `AIResponse`, and `AIClient` protocol.

In [1]:
from dataclasses import dataclass, field
from typing import Protocol, runtime_checkable


@dataclass
class AIRequest:
    """What the business logic sends to the AI backend."""
    prompt: str
    system: str = "You are a helpful assistant."
    temperature: float = 0.0
    max_tokens: int = 256


@dataclass
class AIResponse:
    """What the AI backend returns to the business logic."""
    text: str
    model: str = "unknown"
    usage: dict = field(default_factory=dict)


@runtime_checkable
class AIClient(Protocol):
    """Any AI backend must implement this single method."""
    def complete(self, request: AIRequest) -> AIResponse: ...


print("Shared types ready: AIRequest, AIResponse, AIClient protocol.")

Shared types ready: AIRequest, AIResponse, AIClient protocol.


## Step 2 — The Local Backend: Ollama over HTTP

Ollama exposes a **local HTTP API** at `http://localhost:11434`. This means our local backend looks structurally similar to the hosted one — it's still an HTTP call — but with key differences:

- **No API key** — Ollama runs on your machine, no auth needed
- **No per-token cost** — you pay only for electricity and RAM
- **Data never leaves your machine** — full privacy

In [2]:
import requests


class LocalAIClient:
    """AI backend that calls a local Ollama instance. No API key required."""

    def __init__(
        self,
        model: str = "qwen2.5:0.5b",
        base_url: str = "http://localhost:11434",
        timeout: int = 60,
    ):
        self.model = model
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout

    def complete(self, request: AIRequest) -> AIResponse:
        """Send a request to the local Ollama API."""
        url = f"{self.base_url}/api/chat"

        payload = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": request.system},
                {"role": "user", "content": request.prompt},
            ],
            "stream": False,
            "options": {
                "temperature": request.temperature,
                "num_predict": request.max_tokens,
            },
        }

        # No auth header — Ollama is local, no key needed
        resp = requests.post(url, json=payload, timeout=self.timeout)
        resp.raise_for_status()

        body = resp.json()
        text = body.get("message", {}).get("content", "")
        model_used = body.get("model", self.model)

        return AIResponse(text=text.strip(), model=model_used)


print("LocalAIClient defined.")
print("Key differences from HostedAIClient:")
print("  • No API key parameter")
print("  • No Authorization header")
print("  • Longer default timeout (local inference can be slower on CPU)")

LocalAIClient defined.
Key differences from HostedAIClient:
  • No API key parameter
  • No Authorization header
  • Longer default timeout (local inference can be slower on CPU)


## Step 3 — Verify Protocol Conformance

Same check as before — does it fit the `AIClient` contract?

In [3]:
local = LocalAIClient(model="qwen2.5:0.5b")

print(f"LocalAIClient satisfies AIClient? {isinstance(local, AIClient)}")
print(f"Model:    {local.model}")
print(f"Base URL: {local.base_url}")
print(f"Timeout:  {local.timeout}s")
print(f"API key:  (none — local models don't need one)")

LocalAIClient satisfies AIClient? True
Model:    qwen2.5:0.5b
Base URL: http://localhost:11434
Timeout:  60s
API key:  (none — local models don't need one)


## Step 4 — Call the Local Backend

Let's make a real call. The business logic is identical to what we used with `HostedAIClient` and `MockAIClient`.

In [4]:
request = AIRequest(
    prompt="Classify this support ticket: 'I was charged twice this month'. Reply with one word: billing, technical, account, or other.",
    system="You are a support classifier. Reply with exactly one category name.",
    temperature=0.0,
    max_tokens=10,
)

try:
    response = local.complete(request)
    print(f"✅ Local call succeeded!")
    print(f"   Response: {response.text}")
    print(f"   Model:    {response.model}")
    OLLAMA_AVAILABLE = True
except requests.exceptions.ConnectionError:
    print("⚠️  Ollama is not running on localhost:11434.")
    print("   Start it with: ollama serve")
    print("   Then pull a model: ollama pull qwen2.5:0.5b")
    OLLAMA_AVAILABLE = False
except requests.exceptions.Timeout:
    print("⚠️  Local inference timed out — the model may be too large for your hardware.")
    OLLAMA_AVAILABLE = False

⚠️  Ollama is not running on localhost:11434.
   Start it with: ollama serve
   Then pull a model: ollama pull qwen2.5:0.5b


## Step 5 — The Interface Seam: Same Function, Different Backend

The `classify_ticket` function has **zero knowledge** of whether it's talking to OpenAI's servers or a model running on your laptop. Let's prove it.

In [5]:
def classify_ticket(ticket_text: str, ai_client: AIClient) -> str:
    """Classify a support ticket. The AI backend is injected."""
    categories = ["billing", "technical", "account", "other"]
    req = AIRequest(
        prompt=(
            f"Classify this support ticket into one of {categories}.\n"
            f"Ticket: {ticket_text}\nCategory:"
        ),
        system="Reply with exactly one category name, nothing else.",
        temperature=0.0,
        max_tokens=10,
    )
    resp = ai_client.complete(req)
    category = resp.text.strip().lower().rstrip(".")
    if category not in categories:
        category = "other"
    return category


# Mock backend (always works — no network needed)
class MockAIClient:
    def __init__(self, fixed_answer: str = "billing"):
        self.fixed_answer = fixed_answer
    def complete(self, request: AIRequest) -> AIResponse:
        return AIResponse(text=self.fixed_answer, model="mock-1.0")


mock = MockAIClient()

ticket = "My login keeps failing with error 403"

# Run with mock
print(f"Mock backend:  {classify_ticket(ticket, mock)}")

# Run with local (if available)
if OLLAMA_AVAILABLE:
    print(f"Local backend: {classify_ticket(ticket, local)}")
else:
    print("Local backend: (skipped — Ollama not running)")

print("\n→ Same function, same interface, different backends.")

Mock backend:  billing
Local backend: (skipped — Ollama not running)

→ Same function, same interface, different backends.


## Step 6 — Hosted vs Local: The Comparison Framework

Now let's systematically compare hosted and local backends across five dimensions. This is the **baseline comparison** — a structured way to evaluate tradeoffs before choosing a backend for a project.

In [6]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


comparison = """
## Hosted vs Local Backend Comparison

| Dimension | Hosted (e.g., OpenAI API) | Local (e.g., Ollama) |
|-----------|---------------------------|----------------------|
| **Cost** | Pay per token (~$0.15–$15/M tokens) | Free after hardware — electricity only |
| **Latency** | Network round-trip + inference (~0.5–3s) | No network, but CPU inference can be slow (~2–30s) |
| **Privacy** | Data sent to third-party servers | Data never leaves your machine |
| **Setup** | API key + internet connection | Install Ollama + download model (~0.5–8 GB) |
| **RAM** | None (runs on vendor hardware) | Model loaded in RAM (0.5–16+ GB depending on model) |

### When to Choose Each

| Scenario | Best Choice | Why |
|----------|-------------|-----|
| Production app with high quality needs | **Hosted** | Largest, most capable models |
| Sensitive data (medical, financial, legal) | **Local** | Data never leaves your infrastructure |
| Rapid prototyping | **Hosted** | Fast setup, no hardware concerns |
| Offline or air-gapped environments | **Local** | No internet required |
| High-volume, low-cost batch processing | **Local** | No per-token charges |
| Need the best quality possible | **Hosted** | GPT-4o, Claude 3.5, etc. outperform small local models |
"""

md(comparison)


## Hosted vs Local Backend Comparison

| Dimension | Hosted (e.g., OpenAI API) | Local (e.g., Ollama) |
|-----------|---------------------------|----------------------|
| **Cost** | Pay per token (~$0.15–$15/M tokens) | Free after hardware — electricity only |
| **Latency** | Network round-trip + inference (~0.5–3s) | No network, but CPU inference can be slow (~2–30s) |
| **Privacy** | Data sent to third-party servers | Data never leaves your machine |
| **Setup** | API key + internet connection | Install Ollama + download model (~0.5–8 GB) |
| **RAM** | None (runs on vendor hardware) | Model loaded in RAM (0.5–16+ GB depending on model) |

### When to Choose Each

| Scenario | Best Choice | Why |
|----------|-------------|-----|
| Production app with high quality needs | **Hosted** | Largest, most capable models |
| Sensitive data (medical, financial, legal) | **Local** | Data never leaves your infrastructure |
| Rapid prototyping | **Hosted** | Fast setup, no hardware concerns |
| Offline or air-gapped environments | **Local** | No internet required |
| High-volume, low-cost batch processing | **Local** | No per-token charges |
| Need the best quality possible | **Hosted** | GPT-4o, Claude 3.5, etc. outperform small local models |


## Step 7 — Quick Decision Helper

Let's build a simple decision function that recommends a backend based on project requirements. This makes the comparison concrete and actionable.

In [7]:
def decide_backend(scenario: str) -> str:
    """Simple keyword-based backend recommendation (no AI needed!)."""
    scenario_lower = scenario.lower()

    privacy_keywords = ["hipaa", "gdpr", "sensitive", "medical", "private", "classified", "pii"]
    offline_keywords = ["offline", "air-gapped", "no internet"]
    quality_keywords = ["best quality", "production", "customer-facing", "high accuracy"]
    cost_keywords = ["batch", "high volume", "budget", "cheap", "1000s of"]

    reasons = []

    if any(kw in scenario_lower for kw in privacy_keywords):
        reasons.append(("Local", "Privacy — data stays on your machine"))
    if any(kw in scenario_lower for kw in offline_keywords):
        reasons.append(("Local", "Offline — no network required"))
    if any(kw in scenario_lower for kw in quality_keywords):
        reasons.append(("Hosted", "Quality — largest models available"))
    if any(kw in scenario_lower for kw in cost_keywords):
        reasons.append(("Local", "Cost — no per-token charges at scale"))

    if not reasons:
        reasons.append(("Hosted", "Default — easier setup, good quality"))

    recommendation = reasons[0][0]
    rows = "\n".join(f"| {r[0]} | {r[1]} |" for r in reasons)

    return f"""### Scenario: *{scenario}*

| Recommendation | Reason |
|----------------|--------|
{rows}

**→ Primary recommendation: {recommendation}**
"""


# Run three scenarios
scenarios = [
    "Prototype chatbot for a hackathon demo",
    "Batch-classify 1000s of internal documents on a budget",
    "Customer-facing production app requiring best quality",
]

for s in scenarios:
    md(decide_backend(s))
    print()  # visual separator

### Scenario: *Prototype chatbot for a hackathon demo*

| Recommendation | Reason |
|----------------|--------|
| Hosted | Default — easier setup, good quality |

**→ Primary recommendation: Hosted**


### Scenario: *Batch-classify 1000s of internal documents on a budget*

| Recommendation | Reason |
|----------------|--------|
| Local | Cost — no per-token charges at scale |

**→ Primary recommendation: Local**


### Scenario: *Customer-facing production app requiring best quality*

| Recommendation | Reason |
|----------------|--------|
| Hosted | Quality — largest models available |

**→ Primary recommendation: Hosted**


## Step 8 — The Full Picture: Three Backends, One Protocol

We now have three backends that all satisfy the same `AIClient` interface. The business logic doesn't care which one it gets.

In [8]:
diagram = """
## Architecture: Three Backends, One Interface

```
┌──────────────────────────────────┐
│  Business Logic                  │
│  classify_ticket(text, ai_client)│
│  • builds AIRequest              │
│  • validates AIResponse          │
│  • zero vendor imports           │
└──────────┬───────────────────────┘
           │  ai_client.complete(request)
           │  (the seam)
    ┌──────┴──────┐
    │  AIClient   │  ← Protocol (interface)
    │  Protocol   │
    └──────┬──────┘
     ┌─────┼────────────┐
     │     │            │
     ▼     ▼            ▼
  Mock   Hosted      Local
  (test) (OpenAI)   (Ollama)
  Free   Pay/token   Free
  0 RAM  0 RAM      ~0.5+ GB
  No key API key     No key
```

**Choosing a backend** = changing one constructor call.  
**Business logic** = untouched regardless of which backend runs underneath.
"""

md(diagram)


## Architecture: Three Backends, One Interface

```
┌──────────────────────────────────┐
│  Business Logic                  │
│  classify_ticket(text, ai_client)│
│  • builds AIRequest              │
│  • validates AIResponse          │
│  • zero vendor imports           │
└──────────┬───────────────────────┘
           │  ai_client.complete(request)
           │  (the seam)
    ┌──────┴──────┐
    │  AIClient   │  ← Protocol (interface)
    │  Protocol   │
    └──────┬──────┘
     ┌─────┼────────────┐
     │     │            │
     ▼     ▼            ▼
  Mock   Hosted      Local
  (test) (OpenAI)   (Ollama)
  Free   Pay/token   Free
  0 RAM  0 RAM      ~0.5+ GB
  No key API key     No key
```

**Choosing a backend** = changing one constructor call.  
**Business logic** = untouched regardless of which backend runs underneath.


## 🎯 Summary

### Key Takeaways

1. **Local backend, same protocol** — `LocalAIClient` satisfies the same `AIClient` protocol as `HostedAIClient` and `MockAIClient`, proving the interface seam works across fundamentally different backends
2. **Open vs closed tradeoffs** — hosted models offer superior quality and easy setup; local models offer privacy, zero per-token cost, and offline operation — the right choice depends on the project's constraints
3. **Five-dimension comparison** — systematically evaluating backends on cost, latency, privacy, setup, and RAM prevents ad-hoc decisions and makes the tradeoffs explicit and documented
4. **Swapping is trivial** — because every backend implements `complete(AIRequest) -> AIResponse`, switching from hosted to local (or vice versa) requires changing one constructor call, not rewriting business logic